### 1.Import the libraries and load the dataset

In [7]:
import keras #a high-level deep-learning API that lets you build neural netwroks more easily

from keras.datasets import mnist #MNIST contains images of handwritten digits

from keras.models import Sequential #Sequential is Build my neural network as a sequence of layers

from keras.layers import Dense, Dropout, Flatten
#Imports three types of neural-network layers.
#Dense => create a layer containing n neurons.
#Dropout => randomly deactivate 50% of the neurons during each training step. this help reduce overfitting
#Flatten => turn a multi-dimensional feature map into a 1D vector.


from keras.layers import Conv2D, MaxPooling2D
#Conv2D => This is a convolutional layer, It looks for visual patterns in an image. 
#MaxPooling2D => This reduces the spatial size of the feature maps.

from keras import backend as K #The backend provides lower-level operations that Keras uses internally.

In [8]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(x_train.shape, y_train.shape)

(60000, 28, 28) (60000,)


### 2.Preprocess the data

The image data cannot be fed directly into the model so we need to perform some operations and process the data to make to make it ready for our neural network. The dimension of the training data is (60000,28,28). The CNN model will require one more dimension so we reshape the matrix to shape (60000,28,28,1).

In [ ]:
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)
input_shape = (28, 28, 1)

num_classes = 10

# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

### 3.Create the model

Now we will create our CNN model in Python data science project. A CNN model generally consists of convolutional and pooling layers. It works better for data that are represented as grid structures, this is the reason why CNN works well for image classification problems. The dropout layer is used to deactivate some of the neurons and while training, it reduces offer fitting of the model. We will then compile the model with the Adadelta optimizer.

In [ ]:
from keras.layers import BatchNormalization, Input, RandomRotation, RandomTranslation, RandomZoom

batch_size = 128
num_classes = 10
epochs = 20

model = Sequential()
model.add(Input(shape=input_shape))
# Light augmentation so the model generalizes to hand-drawn digits (which
# are never perfectly centered/scaled like MNIST). These layers are a no-op
# at inference time, so they don't affect the GUI predictions directly.
model.add(RandomRotation(0.08))
model.add(RandomTranslation(0.1, 0.1))
model.add(RandomZoom(0.1))

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss=keras.losses.categorical_crossentropy,optimizer=keras.optimizers.Adam(learning_rate=1e-3),metrics=['accuracy'])

### 4.Train the model

The model.fit() function of Keras will start the training of the model. It takes the training data, validation data, epochs, and batch size.

It takes some time to train the model. After training, we save the weights and model definition in the ‘mnist.h5’ file.

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]

hist = model.fit(x_train, y_train,batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(x_test, y_test),callbacks=callbacks)
print("The model has successfully trained")

model.save('mnist.h5')
print("Saving the model as mnist.h5")

### 5.Evaluate the model

We have 10,000 images in our dataset which will be used to evaluate how good our model works. The testing data was not involved in the training of the data therefore, it is new data for our model. The MNIST dataset is well balanced so we can get around 99% accuracy.

In [14]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

Test loss: 0.6523653864860535
Test accuracy: 0.8504999876022339


### Create GUI to predict digits

In [ ]:
from keras.models import load_model
from tkinter import *
import tkinter as tk

from PIL import Image, ImageDraw, ImageOps
import numpy as np

model = load_model('mnist.h5')

def predict_digit(img):
    img = img.convert('L')
    #MNIST digits are white-on-black; the canvas is drawn black-on-white, so invert
    img = ImageOps.invert(img)
    arr = np.array(img)

    # Crop to the drawn strokes, then rescale/center like MNIST does (digit
    # scaled to fit a ~20px box, centered in the 28x28 frame). Without this,
    # a raw resize of a big off-center stroke looks nothing like training data.
    coords = np.argwhere(arr > 20)
    if coords.size == 0:
        arr28 = np.zeros((28, 28), dtype='float32')
    else:
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        digit = arr[y0:y1, x0:x1]

        h, w = digit.shape
        scale = 20.0 / max(h, w)
        new_h, new_w = max(1, round(h * scale)), max(1, round(w * scale))
        digit_img = Image.fromarray(digit).resize((new_w, new_h), Image.LANCZOS)

        canvas = Image.new('L', (28, 28), 0)
        offset = ((28 - new_w) // 2, (28 - new_h) // 2)
        canvas.paste(digit_img, offset)
        arr28 = np.array(canvas).astype('float32')

    #reshaping to support our model input and normalizing
    arr28 = arr28.reshape(1, 28, 28, 1) / 255.0
    #predicting the class
    res = model.predict([arr28])[0]
    return np.argmax(res), max(res)

class App(tk.Tk):
    def __init__(self):
        tk.Tk.__init__(self)

        self.x = self.y = 0
        self.canvas_size = 300

        # Creating elements
        self.canvas = tk.Canvas(self, width=self.canvas_size, height=self.canvas_size, bg = "white", cursor="cross")
        self.label = tk.Label(self, text="Thinking..", font=("Helvetica", 48))
        self.classify_btn = tk.Button(self, text = "Recognise", command =         self.classify_handwriting)
        self.button_clear = tk.Button(self, text = "Clear", command = self.clear_all)

        # Grid structure
        self.canvas.grid(row=0, column=0, pady=2, sticky=W, )
        self.label.grid(row=0, column=1,pady=2, padx=2)
        self.classify_btn.grid(row=1, column=1, pady=2, padx=2)
        self.button_clear.grid(row=1, column=0, pady=2)

        #self.canvas.bind("<Motion>", self.start_pos)
        self.canvas.bind("<B1-Motion>", self.draw_lines)

        # Drawn in memory alongside the canvas, so we never depend on a screen
        # grab (ImageGrab.grab() is unreliable on Linux/X11/Wayland).
        self.image = Image.new("L", (self.canvas_size, self.canvas_size), 255)
        self.image_draw = ImageDraw.Draw(self.image)

    def clear_all(self):
        self.canvas.delete("all")
        self.image_draw.rectangle([0, 0, self.canvas_size, self.canvas_size], fill=255)

    def classify_handwriting(self):
        digit, acc = predict_digit(self.image)
        self.label.configure(text= str(digit)+', '+ str(int(acc*100))+'%')

    def draw_lines(self, event):
        self.x = event.x
        self.y = event.y
        r=8
        self.canvas.create_oval(self.x-r, self.y-r, self.x + r, self.y + r, fill='black')
        self.image_draw.ellipse([self.x-r, self.y-r, self.x+r, self.y+r], fill=0)

app = App()
mainloop()